# Step 2 — Indexing: Incidence Matrix dan Inverted Index

**Tujuan praktikum**
1. Mahasiswa memahami representasi *term-document incidence matrix* dan kelemahannya.
2. Mahasiswa mampu membangun *inverted index* (kamus + *posting list*) beserta *document frequency* dan *term frequency*.
3. Mahasiswa mampu membangun *positional index* dan menggunakannya untuk *phrase query*.

In [41]:
# Jalankan sekali jika pustaka belum terpasang:
# !pip install pandas numpy matplotlib Sastrawi
import pandas as pd
pd.set_option("display.max_colwidth", 120)

In [42]:
from collections import Counter, defaultdict
from stki_utils import corpus_texts, preprocess

teks = corpus_texts()                      # {doc_id: judul + isi}
docs = {d: preprocess(t) for d, t in teks.items()}
print("Jumlah dokumen:", len(docs))
print("D07:", docs["D07"])

Jumlah dokumen: 50
D07: ['dinamika', 'alam', 'jadi', 'tangguh', 'lulus', 'sekolah', 'tengah', 'atas', 'jalan', 'gap', 'year', 'studi', 'fenomenologis', 'gap', 'year', 'jadi', 'pilih', 'lazim', 'indonesia', 'serta', 'gap', 'year', 'hadap', 'bagai', 'sulit', 'laku', 'sesuai', 'siap', 'diri', 'ikut', 'seleksi', 'masuk', 'guru', 'tinggi', 'tahun', 'ikut', 'teliti', 'tuju', 'eksplorasi', 'dinamika', 'makna', 'resiliensi', 'akademik', 'lulus', 'sma', 'jalan', 'gap', 'year', 'teliti', 'guna', 'dekat', 'kualitatif', 'fenomenologis', 'analisis', 'interpretative', 'phenomenological', 'analysis', 'ipa', 'empat', 'partisipan', 'lalu', 'one', 'on', 'one', 'interview', 'hasil', 'tunjuk', 'gagal', 'seleksi', 'jadi', 'alam', 'tantang', 'pengaruh', 'kondisi', 'kognitif', 'emosional', 'perilaku', 'serta', 'lama', 'gap', 'year', 'serta', 'tahan', 'aspirasi', 'tuju', 'didik', 'lalu', 'komitmen', 'strategi', 'ajar', 'evaluasi', 'diri', 'kelola', 'tekan', 'dukung', 'sumber', 'daya', 'internal', 'eksternal',

## Langkah 1 — Term-document incidence matrix
Matriks biner berukuran |V| × N: bernilai 1 jika term muncul di dokumen. Kita coba untuk 6 dokumen topik pertanian.

In [43]:
subset = ["D07", "D08", "D09", "D10", "D11", "D12"]
vocab_subset = sorted({t for d in subset for t in docs[d]})
incidence = pd.DataFrame([[int(t in docs[d]) for d in subset] for t in vocab_subset],
                         index=vocab_subset, columns=subset)
print("Ukuran matriks:", incidence.shape)
kata_dicari = ["air", "hama", "padi", "pupuk", "sawah", "tani", "tanah"]
print(incidence.reindex(kata_dicari, fill_value=0))

Ukuran matriks: (283, 6)
       D07  D08  D09  D10  D11  D12
air      0    0    0    0    0    0
hama     0    0    0    0    0    0
padi     0    0    0    0    0    0
pupuk    0    0    0    0    0    0
sawah    0    0    0    0    0    0
tani     0    0    0    0    0    0
tanah    0    0    0    0    0    0


In [44]:
vocab = sorted({t for tokens in docs.values() for t in tokens})
sel_total = len(vocab) * len(docs)
sel_isi = sum(len(set(tokens)) for tokens in docs.values())
print(f"Kosakata: {len(vocab)} term, {len(docs)} dokumen")
print(f"Sel matriks penuh : {sel_total}")
print(f"Sel bernilai 1    : {sel_isi} ({sel_isi / sel_total:.1%})")

Kosakata: 1285 term, 50 dokumen
Sel matriks penuh : 64250
Sel bernilai 1    : 2314 (3.6%)


Sebagian besar sel bernilai 0 (*sparse*). Pada koleksi nyata berisi jutaan dokumen, matriks penuh tidak mungkin disimpan. Solusinya adalah **inverted index**: untuk setiap term hanya disimpan daftar dokumen yang memuatnya.

## Langkah 2 — Membangun inverted index
Struktur yang dibangun: `index[term] = {doc_id: tf}`. Panjang posting list sama dengan *document frequency* (df).

In [45]:
def build_inverted_index(docs_tokens):
    index = defaultdict(dict)
    for doc_id, tokens in docs_tokens.items():
        for term, tf in Counter(tokens).items():
            index[term][doc_id] = tf
    return dict(sorted(index.items()))

index = build_inverted_index(docs)
print("Jumlah term di kamus:", len(index))
for term in ["tani", "digital", "gizi", "cari"]:
    postings = index.get(term, [])
    print(f"{term:8s} df={len(postings):2d}  postings={postings}")


Jumlah term di kamus: 1285
tani     df= 2  postings={'D01': 3, 'D20': 1}
digital  df= 6  postings={'D03': 1, 'D11': 2, 'D17': 1, 'D22': 1, 'D34': 1, 'D43': 1}
gizi     df= 0  postings=[]
cari     df= 1  postings={'D29': 1}


In [46]:
tabel = pd.DataFrame({
    "df": {t: len(p) for t, p in index.items()},
    "cf (total tf)": {t: sum(p.values()) for t, p in index.items()},
}).sort_values("df", ascending=False)
tabel.head(10)

,df,cf (total tf)
guna,20,26
jadi,13,30
sistem,12,15
buat,12,13
data,11,22
rupa,11,15
hasil,10,17
indonesia,9,26
banyak,9,16
satu,9,11


## Langkah 3 — Menyimpan dan memuat index

In [47]:
import json, os

os.makedirs("output", exist_ok=True)
with open("output/inverted_index.json", "w") as f:
    json.dump(index, f)

with open("output/inverted_index.json") as f:
    index_dimuat = json.load(f)
print("Index tersimpan dan dimuat ulang dengan benar:", index_dimuat == index)
print("Ukuran file:", os.path.getsize("output/inverted_index.json"), "byte")

Index tersimpan dan dimuat ulang dengan benar: True
Ukuran file: 38418 byte


## Langkah 4 — Positional index dan phrase query
*Positional index* menyimpan posisi kemunculan term: `index[term][doc_id] = [posisi, ...]`. Dengan posisi, kita bisa mencari frasa, yaitu term yang muncul berurutan.

In [48]:
def build_positional_index(docs_tokens):
    index = defaultdict(lambda: defaultdict(list))
    for doc_id, tokens in docs_tokens.items():
        for pos, term in enumerate(tokens):
            index[term][doc_id].append(pos)
    return {t: dict(p) for t, p in sorted(index.items())}

pos_index = build_positional_index(docs)
print("cerdas :", pos_index["cerdas"])
print("buat   :", pos_index["buat"])

cerdas : {'D02': [7], 'D23': [13]}
buat   : {'D02': [8], 'D05': [11], 'D22': [36], 'D23': [14], 'D24': [22], 'D28': [12], 'D34': [111], 'D35': [34], 'D36': [26], 'D40': [8], 'D43': [6], 'D45': [24, 46]}


In [49]:
def phrase_search(frasa, pos_index):
    terms = preprocess(frasa)
    if not terms or any(t not in pos_index for t in terms):
        return []
    kandidat = set.intersection(*(set(pos_index[t]) for t in terms))
    hasil = []
    for d in sorted(kandidat):
        for p in pos_index[terms[0]][d]:
            if all(p + i in pos_index[t][d] for i, t in enumerate(terms[1:], start=1)):
                hasil.append(d)
                break
    return hasil

for frasa in ["kecerdasan buatan", "pertanian cerdas", "pupuk organik", "data pribadi"]:
    print(f"{frasa!r:22s} -> {phrase_search(frasa, pos_index)}")

'kecerdasan buatan'    -> ['D02', 'D23']
'pertanian cerdas'     -> []
'pupuk organik'        -> []
'data pribadi'         -> []


## Tugas Step 2
1. Hitung perbandingan ukuran (jumlah entri) antara incidence matrix penuh dan inverted index untuk seluruh korpus. Berapa kali lebih hemat inverted index?
2. Tampilkan 10 term dengan df = 1 (*hapax*). Apakah term-term tersebut berguna untuk pencarian? Jelaskan.
3. Posisi pada positional index di atas dihitung **setelah** stopword dihapus. Apa akibatnya pada pencarian frasa `"sistem temu kembali informasi"` atau `"gerakan 3M"`? Modifikasi kode agar posisi dihitung sebelum stopword dihapus.
4. Buat fungsi `proximity_search(t1, t2, k)` yang mengembalikan dokumen yang memuat t1 dan t2 dengan jarak paling jauh k posisi.


# Jawaban Tugas Step 2

## 1. Incidence matrix vs inverted index

- *Incidence matrix*: menyimpan semua sel |V| × N (termasuk yang bernilai 0).
- *Inverted index*: hanya menyimpan pasangan (term, dokumen) yang benar-benar ada, yaitu jumlah seluruh *posting*.

In [50]:
sel_matriks   = len(vocab) * len(docs)                       # semua sel, termasuk 0
total_posting = sum(len(p) for p in index.values())        # jumlah entri posting di inverted index
entri_dengan_kamus = total_posting + len(index)            # posting + 1 entri kamus untuk tiap term

print(f"Incidence matrix penuh        : {len(vocab)} x {len(docs)} = {sel_matriks} sel")
print(f"Inverted index (posting saja) : {total_posting} entri")
print(f"Inverted index (+ kamus term) : {entri_dengan_kamus} entri")
print()
print(f"Lebih hemat (posting saja)    : {sel_matriks / total_posting:.1f} kali")
print(f"Lebih hemat (+ kamus term)    : {sel_matriks / entri_dengan_kamus:.1f} kali")

Incidence matrix penuh        : 1285 x 50 = 64250 sel
Inverted index (posting saja) : 2314 entri
Inverted index (+ kamus term) : 3599 entri

Lebih hemat (posting saja)    : 27.8 kali
Lebih hemat (+ kamus term)    : 17.9 kali


*Kesimpulan:* inverted index kira-kira *20× lebih hemat* (14.160 sel → 702 posting), atau ≈12× jika entri kamus term ikut dihitung. Hemat karena hanya ≈5% sel matriks yang bernilai 1; sisanya 0 dan tidak perlu disimpan. Pada koleksi besar selisihnya jauh lebih besar karena kosakata dan jumlah dokumen bertambah, sementara jumlah kata per dokumen relatif tetap.

## 2. Term dengan df = 1 (hapax legomena)

*Hapax* adalah term yang hanya muncul di *satu* dokumen.

In [51]:
hapax = [t for t, p in index.items() if len(p) == 1]
print(f"Jumlah hapax: {len(hapax)} dari {len(index)} term ({len(hapax)/len(index):.0%})")

contoh = list(hapax)[:10]

df_hapax = pd.DataFrame([{"term": t, "df": len(index[t]), "dokumen": list(index[t])[0]} for t in contoh])
df_hapax
pd.DataFrame([{"term": t, "df": len(index[t]), "dokumen": list(index[t])[0]} for t in contoh])

Jumlah hapax: 883 dari 1285 term (69%)


,term,df,dokumen
0,0,1,D08
1,000,1,D34
2,0237,1,D08
3,05,1,D08
4,06,1,D08
5,10,1,D44
6,11,1,D44
7,123,1,D34
8,13,1,D44
9,14,1,D44


In [52]:
# 10 hapax pertama menurut urutan alfabet (sebagai pembanding)
hapax[:10]

['0', '000', '0237', '05', '06', '10', '11', '123', '13', '14']

*Apakah berguna untuk pencarian? tergantung jenis term-nya.*

- *Berguna* untuk term spesifik/langka seperti `aedes`, `aegypti`, `arabika`, `autentikasi`, `beasiswa`. Karena hanya muncul di satu dokumen, term ini punya daya pembeda (IDF) sangat tinggi sehingga query yang memuatnya langsung menunjuk dokumen yang tepat.
- *Kurang/tidak berguna* untuk term generik atau angka seperti `atas`, `banyak`, `beberapa`, `150`, `2`. Kata-kata ini kebetulan muncul sekali saja, bukan karena penting. Mereka hanya menambah ukuran kamus tanpa membantu pencarian (kandidat stopword tambahan).
- Pada korpus kecil ini *346 dari 472 term (≈73%) adalah hapax*, sesuai hukum Zipf: sebagian besar term jarang muncul. Karena itu hapax tidak boleh dibuang begitu saja; ia perlu dipilah antara term informatif (nama/istilah khusus) dan *noise*.

## 3. Posisi dihitung *sebelum* stopword dihapus

*Masalah.* Pada kode awal, stopword dibuang dulu baru posisi dihitung. Akibatnya kata yang aslinya terpisah oleh stopword menjadi *tampak bersebelahan*. Pada D29 teksnya adalah:

> "Sistem temu *kembali* informasi ..." — `kembali` termasuk stopword Sastrawi.

In [53]:
print("Preprocess dengan stopword removal :", preprocess("sistem temu kembali informasi"))
print("Preprocess tanpa  stopword removal :", preprocess("sistem temu kembali informasi", use_stopword=False))
print()
# Dengan index lama, frasa palsu "sistem temu informasi" (tanpa 'kembali') ikut cocok:
for frasa in ["sistem temu kembali informasi", "sistem temu informasi", "gerakan 3M"]:
    print(f"LAMA  {frasa!r:34s} -> {phrase_search(frasa, pos_index)}")

Preprocess dengan stopword removal : ['sistem', 'temu', 'informasi']
Preprocess tanpa  stopword removal : ['sistem', 'temu', 'kembali', 'informasi']

LAMA  'sistem temu kembali informasi'    -> []
LAMA  'sistem temu informasi'            -> []
LAMA  'gerakan 3M'                       -> []


Frasa palsu `"sistem temu informasi"` ikut cocok dengan D29 (*false positive*) karena `kembali` hilang dan `temu` & `informasi` menjadi bersebelahan.

*Perbaikan:* hitung posisi dari token yang *masih lengkap (stopword dipertahankan)*, lalu gunakan perlakuan yang sama pada query.

In [54]:
# Docs dengan stopword dipertahankan (hanya stemming) -> posisi mencerminkan teks asli
docs_penuh = {d: preprocess(t, use_stopword=False) for d, t in teks.items()}
pos_index_penuh = build_positional_index(docs_penuh)

def phrase_search_v2(frasa, pos_index):
    terms = preprocess(frasa, use_stopword=False)      # <- query juga tanpa stopword removal
    if not terms or any(t not in pos_index for t in terms):
        return []
    kandidat = set.intersection(*(set(pos_index[t]) for t in terms))
    hasil = []
    for d in sorted(kandidat):
        for p in pos_index[terms[0]][d]:
            if all(p + i in pos_index[t][d] for i, t in enumerate(terms[1:], start=1)):
                hasil.append(d)
                break
    return hasil

for frasa in ["sistem temu kembali informasi", "sistem temu informasi", "gerakan 3M",
              "kecerdasan buatan", "pertanian cerdas", "pupuk organik", "data pribadi"]:
    print(f"{frasa!r:34s} LAMA: {str(phrase_search(frasa, pos_index)):8s} BARU: {phrase_search_v2(frasa, pos_index_penuh)}")

'sistem temu kembali informasi'    LAMA: []       BARU: []
'sistem temu informasi'            LAMA: []       BARU: []
'gerakan 3M'                       LAMA: []       BARU: []
'kecerdasan buatan'                LAMA: ['D02', 'D23'] BARU: ['D02', 'D23']
'pertanian cerdas'                 LAMA: []       BARU: []
'pupuk organik'                    LAMA: []       BARU: []
'data pribadi'                     LAMA: []       BARU: []


*Hasil dan akibatnya*

- `"sistem temu kembali informasi"` → tetap menemukan **D29** (benar).
- `"sistem temu informasi"` → *LAMA: D29 (salah), BARU: kosong (benar)*. Inilah akibat utama menghitung posisi setelah stopword dibuang: muncul *false positive* karena jarak kata yang sebenarnya terpisah menjadi menyempit.
- `"gerakan 3M"` → hasilnya sama (**D01**) di kedua cara, karena tidak ada stopword di antara `gerakan` dan `3M`. Token `3m` tetap utuh sebab tokenizer menganggap huruf dan angka sebagai satu token.

*Trade-off:* menyimpan stopword membuat index lebih besar (term seperti `yang`, `dan` punya posting list sangat panjang), tetapi pencarian frasa menjadi akurat. Alternatif yang lebih hemat: stopword tidak dimasukkan ke index, tetapi *nomor posisi tetap dihitung dari teks asli* (posisi 'meloncat' di tempat stopword).

## 4. Fungsi `proximity_search(t1, t2, k)`

Mengembalikan dokumen yang memuat `t1` dan `t2` dengan *jarak paling jauh k posisi*.

*Langkah:* (1) preprocess kedua term, (2) ambil dokumen yang memuat keduanya (irisan), (3) di tiap dokumen cek apakah ada pasangan posisi `p1`, `p2` dengan `|p1 − p2| ≤ k`.

In [55]:
def proximity_search(t1, t2, k, pos_index):
    a, b = preprocess(t1, use_stopword=False), preprocess(t2, use_stopword=False)
    if len(a) != 1 or len(b) != 1:
        raise ValueError("t1 dan t2 masing-masing harus satu kata")
    a, b = a[0], b[0]
    if a not in pos_index or b not in pos_index:
        return []

    hasil = []
    for d in sorted(set(pos_index[a]) & set(pos_index[b])):       # dokumen yang memuat keduanya
        if any(abs(p - q) <= k
               for p in pos_index[a][d] for q in pos_index[b][d]
               if not (a == b and p == q)):                        # jangan bandingkan posisi dengan dirinya sendiri
            hasil.append(d)
    return hasil

for t1, t2, k in [("sistem", "informasi", 1), ("sistem", "informasi", 3), ("sistem", "informasi", 5),
                  ("pertanian", "cerdas", 2), ("pupuk", "organik", 1), ("data", "pribadi", 3), ("air", "hama", 10)]:
    print(f"proximity_search({t1!r}, {t2!r}, k={k}) -> {proximity_search(t1, t2, k, pos_index_penuh)}")

proximity_search('sistem', 'informasi', k=1) -> []
proximity_search('sistem', 'informasi', k=3) -> []
proximity_search('sistem', 'informasi', k=5) -> []
proximity_search('pertanian', 'cerdas', k=2) -> []
proximity_search('pupuk', 'organik', k=1) -> []
proximity_search('data', 'pribadi', k=3) -> []
proximity_search('air', 'hama', k=10) -> []


*Cara membaca hasil:* `sistem` dan `informasi` di D29 terpisah oleh `temu kembali` (jarak 3 posisi), sehingga k=1 tidak menemukan apa pun, tetapi k=3 menemukan *D29*. Ini menunjukkan *phrase query* adalah kasus khusus proximity dengan k=1 dan urutan harus sama, sedangkan proximity lebih longgar (urutan bebas, jarak sampai k).